# Trend-Filtered Basis Arbitrage: PAXG vs. XAUT (Two Competing Gold Tokens)

Fifth and final pair in this series: **PAXG** (Paxos's gold token — Paxos Trust Company, regulated
by NYDFS) vs. **XAUT** (Tether Gold — issued by TG Commodities Ltd., a Tether affiliate). Both claim
1:1 physical gold backing with redemption rights, but through **different issuers, different
custodians, different regulatory regimes, and different redemption terms** (minimum lot sizes,
geographic restrictions, KYC requirements all differ between the two).

**A weaker prior than the other four notebooks, stated honestly**: the other pairs in this series
each had a specific, mechanistic reason to expect the *underlying's own trend* to predict the basis —
collateral-demand cycles (WBTC, stETH), or a funding-rate-driven yield mechanism (USDe). Here, the
more plausible driver of any PAXG/XAUT basis is **relative issuer trust and liquidity** (Paxos vs.
Tether have very different regulatory histories and public scrutiny), which isn't really a function
of *gold's* price trend at all. We still test gold's trend against the basis, for methodological
consistency with the rest of this series and because it's a cheap, well-specified test — but go in
expecting a weaker (or absent) relationship here than in the other four notebooks, and treat that as
the informative result if it happens, not a failure of the method.

**Data source**: OKX's public API (free, no key), both instruments. Same real constraint as the
original `paxgGoldArbResearch` notebook: **PAXG's OKX listing only goes back to ~October 2025**,
capping the overlapping sample at roughly the same ~10-month window regardless of which counterparty
we compare it against.

> ⚠️ Research only, not investment advice. Models a market-neutral spread position (long one leg,
> short the other) with no transaction costs, redemption mechanics, or execution slippage.


In [ ]:
# If needed, install dependencies (uncomment to run):
# %pip install pandas numpy matplotlib seaborn scipy statsmodels requests

import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests

pd.set_option("display.float_format", lambda x: f"{x:,.5f}")
sns.set_theme(style="darkgrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)


## 1. Data Collection


In [ ]:
OKX_BASE = "https://www.okx.com/api/v5/market"


def fetch_okx_full_history(inst_id, bar="1D"):
    all_rows = []

    r = requests.get(f"{OKX_BASE}/candles", params={"instId": inst_id, "bar": bar, "limit": 300}, timeout=20)
    r.raise_for_status()
    data = r.json()["data"]
    all_rows.extend(data)
    after = min(int(row[0]) for row in data)

    while True:
        r = requests.get(
            f"{OKX_BASE}/history-candles",
            params={"instId": inst_id, "bar": bar, "limit": 100, "after": after},
            timeout=20,
        )
        r.raise_for_status()
        data = r.json()["data"]
        if not data:
            break
        all_rows.extend(data)
        after = min(int(row[0]) for row in data)
        time.sleep(0.25)

    cols = ["ts", "open", "high", "low", "close", "vol", "volCcy", "volCcyQuote", "confirm"]
    df = pd.DataFrame(all_rows, columns=cols)
    df = df[df["confirm"] == "1"]
    df["date"] = pd.to_datetime(df["ts"].astype(np.int64), unit="ms").dt.normalize()
    df = df.drop_duplicates(subset="date").set_index("date").sort_index()
    for c in ["open", "high", "low", "close", "vol"]:
        df[c] = df[c].astype(float)
    return df[["open", "high", "low", "close", "vol"]]


paxg = fetch_okx_full_history("PAXG-USDT")
xaut = fetch_okx_full_history("XAUT-USDT")

print(f"PAXG-USDT: {paxg.index.min().date()} to {paxg.index.max().date()}, {len(paxg)} days")
print(f"XAUT-USDT: {xaut.index.min().date()} to {xaut.index.max().date()}, {len(xaut)} days")


## 2. Basis Construction

Both legs are USD-quoted separately (unlike the direct cross-rate pairs in the other notebooks), so
the basis is the ratio between them, in percent.


In [ ]:
df = pd.DataFrame({
    "paxg_close": paxg["close"],
    "xaut_close": xaut["close"],
}).dropna()

df["basis_pct"] = df["xaut_close"] / df["paxg_close"] - 1

print(f"Overlapping sample: {df.index.min().date()} to {df.index.max().date()}, {len(df)} days")
df.tail()


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)
axes[0].plot(df.index, df["paxg_close"], label="PAXG", color="tab:blue")
axes[0].plot(df.index, df["xaut_close"], label="XAUT", color="tab:orange", alpha=0.8)
axes[0].set_title("PAXG vs. XAUT — Two Gold-Backed Tokens")
axes[0].legend()

axes[1].plot(df.index, df["basis_pct"] * 100, color="tab:green")
axes[1].axhline(0, color="gray", linewidth=0.8)
axes[1].set_title("Basis: (XAUT / PAXG - 1) x 100%")
axes[1].set_ylabel("%")
plt.tight_layout()
plt.show()

print(f"Basis stats: mean={df['basis_pct'].mean()*100:.4f}%, std={df['basis_pct'].std()*100:.4f}%, "
      f"min={df['basis_pct'].min()*100:.4f}%, max={df['basis_pct'].max()*100:.4f}%")


In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_stat, p_value, *_ = adfuller(df["basis_pct"].dropna())
print(f"ADF test on basis_pct: statistic={adf_stat:.3f}, p-value={p_value:.4f}")
print("p < 0.05 suggests the basis is stationary (mean-reverting) over this sample.")


## 3. Trend Detection (on Gold, via PAXG as the Underlying Proxy)


In [ ]:
def find_pivots(price, window=5):
    is_high = price == price.rolling(window * 2 + 1, center=True).max()
    is_low = price == price.rolling(window * 2 + 1, center=True).min()
    return price[is_high], price[is_low]


def fit_trendline(pivot_points, n_recent=3):
    if len(pivot_points) < 2:
        return None
    recent = pivot_points.iloc[-n_recent:]
    x = (recent.index - recent.index[0]).days.values.astype(float)
    y = recent.values
    slope, intercept = np.polyfit(x, y, 1)
    return recent.index[0], slope, intercept


swing_highs, swing_lows = find_pivots(df["paxg_close"], window=5)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(df.index, df["paxg_close"], color="tab:blue", label="PAXG close", zorder=1)
ax.scatter(swing_highs.index, swing_highs.values, color="red", s=25, zorder=3, label="swing highs")
ax.scatter(swing_lows.index, swing_lows.values, color="green", s=25, zorder=3, label="swing lows")

for pivots, color in [(swing_highs, "red"), (swing_lows, "green")]:
    fit = fit_trendline(pivots, n_recent=4)
    if fit:
        x0, slope, intercept = fit
        x_line = pd.date_range(x0, df.index[-1])
        y_line = intercept + slope * (x_line - x0).days
        ax.plot(x_line, y_line, color=color, linestyle="--", linewidth=1.5, alpha=0.8)

ax.set_title("Gold (via PAXG) Price with Recent Swing-High / Swing-Low Trend Lines")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
def rolling_trend_pct(price, window=20):
    def slope(y):
        x = np.arange(len(y))
        return np.polyfit(x, y, 1)[0]
    raw_slope = price.rolling(window).apply(slope, raw=True)
    return raw_slope * window / price

df["trend_pct"] = rolling_trend_pct(df["paxg_close"], window=20)

fig, ax = plt.subplots()
df["trend_pct"].mul(100).plot(ax=ax, color="purple")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_title("Rolling 20-Day Gold Trend Signal (Implied %% Move Over Window)")
ax.set_ylabel("%")
plt.tight_layout()
plt.show()


## 4. Does Gold's Trend Predict the PAXG/XAUT Basis?

Going in with a weaker prior than the other notebooks — the more plausible driver here is relative
issuer trust/liquidity, not gold's price action — so a near-zero result is the *expected* outcome,
not a null result to be surprised by.


In [ ]:
def zscore(s, window=20):
    return (s - s.rolling(window).mean()) / s.rolling(window).std()

df["basis_z"] = zscore(df["basis_pct"], window=20)

sub = df[["trend_pct", "basis_z"]].dropna()
corr_val = sub["trend_pct"].corr(sub["basis_z"])

fig, ax = plt.subplots()
ax.scatter(sub["trend_pct"] * 100, sub["basis_z"], alpha=0.5, s=20)
ax.set_xlabel("Rolling gold trend signal (%% move over 20d)")
ax.set_ylabel("PAXG/XAUT basis z-score")
ax.set_title(f"Gold Trend vs. PAXG/XAUT Basis Deviation (n={len(sub)}, corr={corr_val:.3f})")
ax.axhline(0, color="gray", linewidth=0.8)
ax.axvline(0, color="gray", linewidth=0.8)
plt.tight_layout()
plt.show()

print(f"Correlation(trend_pct, basis_z) = {corr_val:.3f}")
if abs(corr_val) < 0.15:
    print("Near zero, consistent with the weaker prior stated above: this basis looks more like an "
          "issuer-trust/liquidity phenomenon than something gold's own trend explains.")
elif corr_val > 0:
    print("Positive — unexpected given the weaker prior. Worth investigating what's actually driving "
          "this before trusting it; may not be a gold-trend effect at all (e.g. could reflect "
          "correlated flows into 'risk-on' tokens broadly during crypto-wide uptrends).")
else:
    print("Negative — also unexpected given the weaker prior. Same caveat: investigate the actual "
          "driver before trusting it.")


## 5. Mean-Reversion Arbitrage Signal

**Position convention**: `+1` = long the basis (long XAUT / short PAXG, taken when XAUT is *cheap*
relative to PAXG, relative to their own recent history). `-1` = short the basis (short XAUT / long
PAXG, taken when XAUT is rich relative to PAXG).


In [ ]:
def build_positions(z, entry=2.0, exit=0.5, blocked=None):
    position = pd.Series(0.0, index=z.index)
    current = 0.0
    for i, (idx, val) in enumerate(z.items()):
        if pd.isna(val):
            position.iloc[i] = current
            continue
        is_blocked = bool(blocked.loc[idx]) if blocked is not None and idx in blocked.index else False
        if current == 0.0:
            if not is_blocked:
                if val > entry:
                    current = -1.0
                elif val < -entry:
                    current = 1.0
        else:
            if abs(val) < exit:
                current = 0.0
        position.iloc[i] = current
    return position


ENTRY_Z, EXIT_Z = 2.0, 0.5
position_pure_mr = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z)

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
df["basis_z"].plot(ax=axes[0], color="tab:green")
axes[0].axhline(ENTRY_Z, color="red", linestyle="--", linewidth=0.8)
axes[0].axhline(-ENTRY_Z, color="red", linestyle="--", linewidth=0.8)
axes[0].axhline(0, color="gray", linewidth=0.8)
axes[0].set_title("PAXG/XAUT Basis Z-Score with Entry Thresholds")

position_pure_mr.plot(ax=axes[1], color="black", drawstyle="steps-post")
axes[1].set_title("Pure Mean-Reversion Position (+1 = long basis, -1 = short basis)")
plt.tight_layout()
plt.show()


## 6. Combining Both Ideas: Trend-Filtered Arbitrage

Included for consistency with the rest of the series, even though Section 4's weaker prior means we
shouldn't expect this to help much here specifically.


In [ ]:
trend_strength = df["trend_pct"].abs()
trend_threshold = trend_strength.rolling(90, min_periods=30).quantile(0.67)
strong_trend = trend_strength > trend_threshold

position_trend_filtered = build_positions(df["basis_z"], entry=ENTRY_Z, exit=EXIT_Z, blocked=strong_trend)

print(f"Days blocked by trend filter: {strong_trend.sum()} / {strong_trend.notna().sum()} "
      f"({strong_trend.mean()*100:.1f}%)")


In [ ]:
def sharpe(returns, periods_per_year=365):
    r = returns.dropna()
    if len(r) == 0 or r.std() == 0:
        return np.nan
    return (r.mean() / r.std()) * np.sqrt(periods_per_year)


def max_drawdown(cum_returns):
    running_max = cum_returns.cummax()
    dd = cum_returns / running_max - 1
    return dd.min()


basis_chg = df["basis_pct"].diff()

strategies = pd.DataFrame({
    "pure_mean_reversion": position_pure_mr.shift(1) * basis_chg,
    "trend_filtered_mean_reversion": position_trend_filtered.shift(1) * basis_chg,
})

cum = (1 + strategies.fillna(0)).cumprod()
cum.plot(title="Cumulative Return: Pure vs. Trend-Filtered PAXG/XAUT Basis Arbitrage")
plt.tight_layout()
plt.show()

summary = pd.DataFrame({
    "annualized_sharpe": strategies.apply(sharpe),
    "total_return": cum.iloc[-1] - 1,
    "annualized_vol": strategies.std() * np.sqrt(365),
    "max_drawdown": cum.apply(max_drawdown),
    "n_trades": pd.Series({
        "pure_mean_reversion": (position_pure_mr.diff().abs() > 0).sum(),
        "trend_filtered_mean_reversion": (position_trend_filtered.diff().abs() > 0).sum(),
    }),
})
summary


## 7. Market-Neutrality Check


In [ ]:
gold_ret = np.log(df["paxg_close"]).diff()

exposure_check = pd.DataFrame({
    "pure_mr_vs_gold": [strategies["pure_mean_reversion"].corr(gold_ret)],
    "trend_filtered_vs_gold": [strategies["trend_filtered_mean_reversion"].corr(gold_ret)],
}, index=["correlation_with_gold_return"])
exposure_check


## 8. Does the Gross Edge Survive Real Trading Costs?


In [ ]:
TAKER_FEE_PER_FILL = 0.0005
FILLS_PER_ROUND_TRIP = 4

fee_drag = summary["n_trades"] * FILLS_PER_ROUND_TRIP * TAKER_FEE_PER_FILL
net_return_estimate = summary["total_return"] - fee_drag

cost_comparison = pd.DataFrame({
    "gross_total_return": summary["total_return"],
    "est_fee_drag": fee_drag,
    "est_net_total_return": net_return_estimate,
})
cost_comparison


## 9. Takeaways & Next Steps — And Across All Five Notebooks in This Series

- **This pair's result should be read against its weaker prior** (Section 4): unlike WBTC, stETH, and
  USDe, there wasn't a strong mechanistic reason to expect gold's trend to predict this specific
  basis going in. A near-zero correlation here is confirmation the method is behaving sensibly, not a
  disappointing result.
- **Same short-history constraint as the original PAXG/XAU-perp notebook**: PAXG's OKX listing start
  (~October 2025) bounds this analysis to ~10 months regardless of counterparty — re-run as more
  history accumulates.
- **Zoom out across all five notebooks in this series** (PAXG/XAU-perp, stETH/ETH, WBTC/BTC, USDe,
  and this one): the trend-basis correlation came back **near zero or opposite-of-hypothesis in four
  of five pairs**, and only genuinely confirmed the hypothesis for **USDe** — the one pair whose
  economic mechanism (funding-rate-driven yield) most directly ties the underlying's trend to the
  spread's health. That's not a coincidence to wave away: it suggests "trend explains basis" needs a
  *specific, mechanistic* reason to hold, not just structural similarity between two instruments
  tracking the same asset. And even where the correlation held (USDe), the trend filter still didn't
  improve the backtest, and **every single pair's gross edge was small enough to be erased by
  conservative trading-fee assumptions**. If there's a headline finding across this whole series,
  it's that: these bases are real and often statistically mean-reverting, but not exploitable at
  retail-fee-tier trading costs without either a much better fee structure, a longer holding horizon
  per trade, or a stronger, more selective entry signal than a plain z-score threshold.
- **Robustness**: validate out-of-sample and account for transaction costs (Section 8) before
  treating any of this as an actual edge.
